# Transfer Learning con texto: solo la cabeza vs. Fine-tuning

## Objetivo
Comparar dos formas de aprendizaje por transferencia con un modelo de lenguaje preentrenado (DistilBERT):

1. **Solo la cabeza:** congelar DistilBERT y entrenar únicamente el clasificador nuevo.
2. **Fine-tuning:** permitir que se actualicen los pesos de DistilBERT y entrenar también el clasificador.

Usaremos frases en inglés con etiquetas de sentimiento: `1 = positivo`, `0 = negativo`.

**Nota didáctica:** el conjunto de datos es pequeño y artificial. Sirve para entender el procedimiento, no para medir el rendimiento real de un sistema de análisis de sentimientos. La primera ejecución descarga DistilBERT desde Hugging Face y necesita conexión a Internet.

## Paso 1. Instalar las bibliotecas

Ejecuta esta celda primero. En Google Colab, PyTorch suele venir instalado; instalamos `transformers` para descargar y utilizar DistilBERT.

In [ ]:
!pip -q install transformers

## Paso 2. Importar las bibliotecas y fijar la semilla

La semilla ayuda a que la parte aleatoria del experimento sea más reproducible. Los resultados pueden variar entre entornos.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMILLA)

dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', dispositivo)

## Paso 3. Crear el conjunto de datos

Cada frase tiene una etiqueta: `1` para positiva y `0` para negativa. Usaremos exactamente la misma partición para los dos métodos.

In [ ]:
textos = [
    'I love this movie',
    'This film is wonderful',
    'The food was delicious',
    'I am very happy',
    'This is an excellent product',
    'The experience was amazing',
    'I really enjoyed it',
    'Everything was fantastic',
    'I hate this movie',
    'This film is terrible',
    'The food was disgusting',
    'I am very disappointed',
    'This product is awful',
    'The experience was horrible',
    'I really disliked it',
    'Everything was bad',
    'The service was excellent',
    'What a wonderful day',
    'The product broke immediately',
    'I regret buying this',
    'I am delighted with the result',
    'This was a complete disaster',
    'The quality is fantastic',
    'I would never recommend it'
]

# 1 = positivo; 0 = negativo
etiquetas = np.array([
    1, 1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0, 0, 0,
    1, 1, 0, 0, 1, 0, 1, 0
], dtype=np.float32)

# Partición fija para que ambos experimentos utilicen los mismos datos
train_texts = textos[:16]
train_labels = etiquetas[:16]
test_texts = textos[16:]
test_labels = etiquetas[16:]

print('Frases totales:', len(textos))
print('Entrenamiento:', len(train_texts))
print('Prueba:', len(test_texts))
print('Etiquetas de entrenamiento:', train_labels.astype(int))

## Paso 4. Descargar el tokenizador y el modelo preentrenado

DistilBERT ya aprendió representaciones del inglés durante su entrenamiento previo. El tokenizador convierte las frases en tokens numéricos que el modelo puede procesar.

Esta celda necesita conexión a Internet la primera vez.

In [ ]:
modelo_nombre = 'distilbert/distilbert-base-uncased'
tokenizador = AutoTokenizer.from_pretrained(modelo_nombre)

X_train = tokenizador(
    train_texts, padding=True, truncation=True,
    max_length=64, return_tensors='pt'
)
X_test = tokenizador(
    test_texts, padding=True, truncation=True,
    max_length=64, return_tensors='pt'
)

print('Forma de input_ids para entrenamiento:', tuple(X_train['input_ids'].shape))
print('Forma de input_ids para prueba:', tuple(X_test['input_ids'].shape))
print('\nEjemplo de frase:', train_texts[0])
print('Tokens:', tokenizador.tokenize(train_texts[0]))
print('IDs numéricos:', X_train['input_ids'][0].tolist())

## Paso 5. Definir el modelo con una cabeza de clasificación

La salida de DistilBERT contiene un vector por token. Para obtener un vector por frase, promediamos las representaciones de los tokens reales e ignoramos los tokens de relleno mediante la máscara de atención. La cabeza `Linear` convierte ese vector en una puntuación para la clase positiva.

- `congelar=True`: los pesos de DistilBERT no cambian.
- `congelar=False`: los pesos de DistilBERT pueden cambiar.

In [ ]:
class ClasificadorSentimiento(nn.Module):
    def __init__(self, congelar=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(modelo_nombre)
        self.congelar = congelar
        self.cabeza = nn.Linear(self.encoder.config.hidden_size, 1)

        for parametro in self.encoder.parameters():
            parametro.requires_grad = not congelar

    def forward(self, input_ids, attention_mask):
        if self.congelar:
            with torch.no_grad():
                salida = self.encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )
        else:
            salida = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        vectores = salida.last_hidden_state
        mascara = attention_mask.unsqueeze(-1).float()
        vector_frase = (vectores * mascara).sum(dim=1) / mascara.sum(dim=1).clamp(min=1)
        logits = self.cabeza(vector_frase).squeeze(-1)
        return logits

def contar_parametros(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    print(f'Parámetros totales: {total:,}')
    print(f'Parámetros entrenables: {entrenables:,}')
    print(f'Porcentaje entrenable: {100*entrenables/total:.3f}%')

## Paso 6. Definir una función de entrenamiento

La función muestra la pérdida por época. Para el fine-tuning utilizamos una tasa de aprendizaje pequeña para el modelo preentrenado y una mayor para la cabeza nueva.

In [ ]:
def entrenar_modelo(congelar, epocas=3, batch_size=4):
    modelo = ClasificadorSentimiento(congelar=congelar).to(dispositivo)
    contar_parametros(modelo)

    if congelar:
        grupos = [{'params': modelo.cabeza.parameters(), 'lr': 1e-3}]
    else:
        grupos = [
            {'params': modelo.encoder.parameters(), 'lr': 2e-5},
            {'params': modelo.cabeza.parameters(), 'lr': 1e-3}
        ]

    optimizador = torch.optim.AdamW(grupos)
    funcion_perdida = nn.BCEWithLogitsLoss()

    conjunto = TensorDataset(
        X_train['input_ids'], X_train['attention_mask'], torch.tensor(train_labels)
    )
    generador = torch.Generator().manual_seed(SEMILLA)
    cargador = DataLoader(conjunto, batch_size=batch_size, shuffle=True, generator=generador)
    historial = []

    for epoca in range(epocas):
        modelo.train()
        if congelar:
            # Mantener DistilBERT en modo evaluación y congelado.
            modelo.encoder.eval()

        perdida_acumulada = 0.0
        for ids, mascara, y in cargador:
            ids = ids.to(dispositivo)
            mascara = mascara.to(dispositivo)
            y = y.to(dispositivo)

            optimizador.zero_grad()
            logits = modelo(ids, mascara)
            perdida = funcion_perdida(logits, y)
            perdida.backward()
            optimizador.step()
            perdida_acumulada += perdida.item()

        promedio = perdida_acumulada / len(cargador)
        historial.append(promedio)
        print(f'Época {epoca+1}/{epocas} | pérdida: {promedio:.4f}')

    return modelo, historial

## Paso 7. Experimento A: entrenar solo la cabeza

DistilBERT conserva sus pesos preentrenados. Solo se actualizan los parámetros de la capa de clasificación.

In [ ]:
modelo_A, perdida_A = entrenar_modelo(
    congelar=True,
    epocas=3
)


## Paso 8. Experimento B: fine-tuning

Creamos otra instancia nueva de DistilBERT con los pesos preentrenados originales. Esta vez, los pesos del encoder y los de la cabeza pueden actualizarse. No reutilizamos el encoder modificado por el experimento A.

In [ ]:
modelo_B, perdida_B = entrenar_modelo(
    congelar=False,
    epocas=3
)


## Paso 9. Evaluar ambos modelos con las mismas frases de prueba

La exactitud es la proporción de predicciones correctas. Debido al tamaño diminuto del conjunto de prueba, debe interpretarse solo como demostración.

In [ ]:
def evaluar_modelo(modelo, nombre):
    modelo.eval()
    ids = X_test['input_ids'].to(dispositivo)
    mascara = X_test['attention_mask'].to(dispositivo)
    with torch.no_grad():
        logits = modelo(ids, mascara)
        probabilidades = torch.sigmoid(logits).cpu().numpy()

    predicciones = (probabilidades >= 0.5).astype(int)
    exactitud = (predicciones == test_labels.astype(int)).mean()
    print(f'\n{nombre} | exactitud de prueba: {exactitud:.1%}')
    for frase, prob, pred, real in zip(test_texts, probabilidades, predicciones, test_labels):
        clase = 'POSITIVO' if pred == 1 else 'NEGATIVO'
        etiqueta_real = 'POSITIVO' if real == 1 else 'NEGATIVO'
        print(f'{frase:36s} -> {clase:8s} (p={prob:.3f}; real={etiqueta_real})')
    return exactitud

acc_A = evaluar_modelo(modelo_A, 'A. SOLO LA CABEZA')
acc_B = evaluar_modelo(modelo_B, 'B. FINE-TUNING')

## Paso 10. Comparar las curvas de pérdida

Una pérdida menor en entrenamiento no garantiza mejor generalización. Para una comparación rigurosa se necesitarían más datos y un conjunto de validación separado.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(perdida_A)+1), perdida_A, marker='o', label='Solo la cabeza')
plt.plot(range(1, len(perdida_B)+1), perdida_B, marker='s', label='Fine-tuning')
plt.xlabel('Época')
plt.ylabel('Pérdida de entrenamiento')
plt.title('Comparación de las dos estrategias')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f'Exactitud solo la cabeza: {acc_A:.1%}')
print(f'Exactitud fine-tuning:    {acc_B:.1%}')

## Conclusiones

- **Solo la cabeza:** el modelo preentrenado extrae características fijas; solo aprende el clasificador nuevo. Es normalmente más rápido y requiere menos memoria.
- **Fine-tuning:** el modelo preentrenado y la cabeza pueden adaptarse a la tarea. Suele requerir más tiempo y puede sobreajustarse si hay pocos ejemplos.
- En ambos casos se parte de pesos preentrenados, no de pesos aleatorios para el encoder.
- Este experimento es ilustrativo: 24 frases no bastan para concluir qué método es mejor en un problema real.

**Para ejecutar:** en Google Colab, usa `Entorno de ejecución > Ejecutar todas` y espera a que termine cada celda. La primera ejecución descargará el modelo.